In [ ]:
# Import Library
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import matplotlib.pyplot as plt

In [ ]:
# up dataset
from google.colab import files
uploaded = files.upload()

Saving PhiUSIIL_Phishing_URL_Dataset.csv to PhiUSIIL_Phishing_URL_Dataset.csv


In [ ]:
# Load CSV
import pandas as pd

df = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset.csv')
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [ ]:
# ngehapus duplicate url
print("Before removing duplicate URLs:", df.shape)
print("Duplicate URLs:", df['URL'].duplicated().sum())

df = df.drop_duplicates(subset='URL', keep='first').reset_index(drop=True)

print("After removing duplicate URLs:", df.shape)
print("Duplicate URLs:", df['URL'].duplicated().sum())

Before removing duplicate URLs: (235795, 56)
Duplicate URLs: 425
After removing duplicate URLs: (235370, 56)
Duplicate URLs: 0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235370 entries, 0 to 235369
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235370 non-null  object 
 1   URL                         235370 non-null  object 
 2   URLLength                   235370 non-null  int64  
 3   Domain                      235370 non-null  object 
 4   DomainLength                235370 non-null  int64  
 5   IsDomainIP                  235370 non-null  int64  
 6   TLD                         235370 non-null  object 
 7   URLSimilarityIndex          235370 non-null  float64
 8   CharContinuationRate        235370 non-null  float64
 9   TLDLegitimateProb           235370 non-null  float64
 10  URLCharProb                 235370 non-null  float64
 11  TLDLength                   235370 non-null  int64  
 12  NoOfSubDomain               235370 non-null  int64  
 13  HasObfuscation

In [ ]:
df.shape

(235370, 56)

In [ ]:
df.columns

Index(['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP',
       'TLD', 'URLSimilarityIndex', 'CharContinuationRate',
       'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain',
       'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
       'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL',
       'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength',
       'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore',
       'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect',
       'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame',
       'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton',
       'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto',
       'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef',
       'NoOfEmptyRef', 'NoOf

In [ ]:
df['label'].value_counts()

,count
label,
1,134850
0,100520


In [ ]:
# preprocessing buat menghapus kolom teks, X dan y, sekaligus menghapus data duplikat
drop_cols = [
    'FILENAME',
    'URL',
    'Domain',
    'TLD',
    'Title'
]


feature_cols = [
    column for column in df.columns
    if column not in drop_cols + ['label']
]

print("Jumlah data sebelum penghapusan duplikat fitur:", len(df))


duplicate_feature_rows = df[
    df.duplicated(
        subset=feature_cols,
        keep=False
    )
].copy()

if len(duplicate_feature_rows) > 0:

    label_check = (
        duplicate_feature_rows
        .groupby(
            feature_cols,
            dropna=False
        )['label']
        .nunique()
    )

    conflicting_patterns = label_check[
        label_check > 1
    ]

    print(
        "Jumlah pola fitur identik dengan label berbeda:",
        len(conflicting_patterns)
    )

    if len(conflicting_patterns) > 0:
        raise ValueError(
            "Ditemukan kombinasi fitur yang sama tetapi labelnya berbeda. "
            "Data tersebut perlu diperiksa sebelum dilanjutkan."
        )

df = (
    df
    .drop_duplicates(
        subset=feature_cols,
        keep='first'
    )
    .reset_index(drop=True)
)

print("Jumlah data setelah penghapusan duplikat fitur:", len(df))
print(
    "Jumlah data yang dihapus:",
    len(duplicate_feature_rows) -
    duplicate_feature_rows.drop_duplicates(
        subset=feature_cols
    ).shape[0]
)

X = df[feature_cols].copy()
y = df['label'].copy()

print("\nJumlah fitur input:", X.shape[1])
print("Jumlah data:", X.shape[0])

print("\nDistribusi kelas:")
print(y.value_counts().sort_index())

print(
    "\nSisa duplikat berdasarkan seluruh fitur:",
    X.duplicated().sum()
)

assert X.duplicated().sum() == 0

print("\nDuplikat fitur berhasil dihapus.")

Jumlah data sebelum penghapusan duplikat fitur: 235370
Jumlah pola fitur identik dengan label berbeda: 0
Jumlah data setelah penghapusan duplikat fitur: 234562
Jumlah data yang dihapus: 808

Jumlah fitur input: 50
Jumlah data: 234562

Distribusi kelas:
label
0     99712
1    134850
Name: count, dtype: int64

Sisa duplikat berdasarkan seluruh fitur: 0

Duplikat fitur berhasil dihapus.


In [ ]:
# pembagian data antara train, vald, test
from sklearn.model_selection import train_test_split

# tahap pertama, 70% train dan 30% data sementara
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# tahap kedua, 30% data sementara dibagi dua: 15% valid dan 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("DISTRIBUSI DATA SETELAH SPLIT")
print("=" * 50)

print(
    f"Training   : {len(y_train):,} data "
    f"({len(y_train) / len(y):.2%})"
)
print(y_train.value_counts().sort_index())

print(
    f"\nValidation : {len(y_val):,} data "
    f"({len(y_val) / len(y):.2%})"
)
print(y_val.value_counts().sort_index())

print(
    f"\nTesting    : {len(y_test):,} data "
    f"({len(y_test) / len(y):.2%})"
)
print(y_test.value_counts().sort_index())

DISTRIBUSI DATA SETELAH SPLIT
Training   : 164,193 data (70.00%)
label
0    69798
1    94395
Name: count, dtype: int64

Validation : 35,184 data (15.00%)
label
0    14957
1    20227
Name: count, dtype: int64

Testing    : 35,185 data (15.00%)
label
0    14957
1    20228
Name: count, dtype: int64


In [ ]:
# balancing train pakai Random Oversampling
from imblearn.over_sampling import RandomOverSampler

random_oversampler = RandomOverSampler(
    random_state=42
)

X_train_balanced, y_train_balanced = (
    random_oversampler.fit_resample(
        X_train,
        y_train
    )
)

print("DISTRIBUSI TRAINING SEBELUM OVERSAMPLING")
print("=" * 50)
print(y_train.value_counts().sort_index())

print("\nDISTRIBUSI TRAINING SETELAH OVERSAMPLING")
print("=" * 50)
print(y_train_balanced.value_counts().sort_index())

print("\nDISTRIBUSI VALIDATION")
print("=" * 50)
print(y_val.value_counts().sort_index())

print("\nDISTRIBUSI TEST")
print("=" * 50)
print(y_test.value_counts().sort_index())

# Memastikan training hasil oversampling sudah seimbang
assert y_train_balanced.value_counts().nunique() == 1

# Memastikan validation dan test tidak berubah
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

print("\nRandom Oversampling hanya diterapkan pada training.")
print("Validation dan test tetap menggunakan data asli.")

DISTRIBUSI TRAINING SEBELUM OVERSAMPLING
label
0    69798
1    94395
Name: count, dtype: int64

DISTRIBUSI TRAINING SETELAH OVERSAMPLING
label
0    94395
1    94395
Name: count, dtype: int64

DISTRIBUSI VALIDATION
label
0    14957
1    20227
Name: count, dtype: int64

DISTRIBUSI TEST
label
0    14957
1    20228
Name: count, dtype: int64

Random Oversampling hanya diterapkan pada training.
Validation dan test tetap menggunakan data asli.


In [ ]:
# ngedefinisikan 5 model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    )
}

print("Model yang digunakan:")
for name in models:
    print("-", name)

Model yang digunakan:
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- AdaBoost


In [ ]:
# train, vald, test, dan pengukuran waktu
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

results = []

for name, model in models.items():

    print(f"Melatih model: {name}")


    train_start = time.perf_counter()

    model.fit(
        X_train_balanced,
        y_train_balanced
    )

    train_time = time.perf_counter() - train_start


    y_val_pred = model.predict(X_val)


    prediction_times = []

    for _ in range(10):

        predict_start = time.perf_counter()

        y_test_pred = model.predict(X_test)

        prediction_time = (
            time.perf_counter() - predict_start
        )

        prediction_times.append(prediction_time)

    average_prediction_time = np.mean(
        prediction_times
    )


    inference_ms_per_sample = (
        average_prediction_time / len(X_test)
    ) * 1000


    results.append({
        "Model": name,

        "Val Accuracy": accuracy_score(
            y_val,
            y_val_pred
        ),

        "Val Precision": precision_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0
        ),

        "Val Recall": recall_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0
        ),

        "Val F1-score": f1_score(
            y_val,
            y_val_pred,
            average="weighted",
            zero_division=0
        ),

        "Test Accuracy": accuracy_score(
            y_test,
            y_test_pred
        ),

        "Test Precision": precision_score(
            y_test,
            y_test_pred,
            average="weighted",
            zero_division=0
        ),

        "Test Recall": recall_score(
            y_test,
            y_test_pred,
            average="weighted",
            zero_division=0
        ),

        "Test F1-score": f1_score(
            y_test,
            y_test_pred,
            average="weighted",
            zero_division=0
        ),

        "Training Time (s)": train_time,

        "Prediction Time (s)": (
            average_prediction_time
        ),

        "Inference (ms/sample)": (
            inference_ms_per_sample
        )
    })


results_df = pd.DataFrame(results)


results_df = results_df.sort_values(
    by=[
        "Val F1-score",
        "Inference (ms/sample)"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

results_df

Melatih model: Logistic Regression
Melatih model: Decision Tree
Melatih model: Random Forest
Melatih model: XGBoost
Melatih model: AdaBoost


,Model,Val Accuracy,Val Precision,Val Recall,Val F1-score,Test Accuracy,Test Precision,Test Recall,Test F1-score,Training Time (s),Prediction Time (s),Inference (ms/sample)
0,Decision Tree,0.999943,0.999943,0.999943,0.999943,1.000000,1.000000,1.000000,1.000000,0.716434,0.010321,0.000293
1,XGBoost,0.999943,0.999943,0.999943,0.999943,1.000000,1.000000,1.000000,1.000000,2.601584,0.110625,0.003144
2,Random Forest,0.999943,0.999943,0.999943,0.999943,1.000000,1.000000,1.000000,1.000000,20.701628,0.170290,0.004840
3,AdaBoost,0.999915,0.999915,0.999915,0.999915,1.000000,1.000000,1.000000,1.000000,17.999671,0.199752,0.005677
4,Logistic Regression,0.999886,0.999886,0.999886,0.999886,0.999915,0.999915,0.999915,0.999915,1.023498,0.016711,0.000475


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

for name, model in models.items():

    # Prediksi menggunakan test asli
    y_test_pred = model.predict(X_test)

    print("=" * 60)
    print(f"MODEL: {name}")
    print("=" * 60)

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            y_test_pred
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_test_pred,
            digits=6,
            zero_division=0
        )
    )

    print()

MODEL: Logistic Regression

Confusion Matrix:
[[14954     3]
 [    0 20228]]

Classification Report:
              precision    recall  f1-score   support

           0   1.000000  0.999799  0.999900     14957
           1   0.999852  1.000000  0.999926     20228

    accuracy                       0.999915     35185
   macro avg   0.999926  0.999900  0.999913     35185
weighted avg   0.999915  0.999915  0.999915     35185


MODEL: Decision Tree

Confusion Matrix:
[[14957     0]
 [    0 20228]]

Classification Report:
              precision    recall  f1-score   support

           0   1.000000  1.000000  1.000000     14957
           1   1.000000  1.000000  1.000000     20228

    accuracy                       1.000000     35185
   macro avg   1.000000  1.000000  1.000000     35185
weighted avg   1.000000  1.000000  1.000000     35185


MODEL: Random Forest

Confusion Matrix:
[[14957     0]
 [    0 20228]]

Classification Report:
              precision    recall  f1-score   support

In [ ]:
# feature importance
rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(10)

,Feature,Importance
3,URLSimilarityIndex,0.183686
49,NoOfExternalRef,0.179250
22,LineOfCode,0.149019
47,NoOfSelfRef,0.111082
44,NoOfImage,0.080794
46,NoOfJS,0.058853
36,HasSocialNet,0.049907
43,HasCopyrightInfo,0.027220
23,LargestLineLength,0.019635
32,HasDescription,0.019142


In [ ]:
train_urls = set(
    df.loc[X_train.index, "URL"]
)

val_urls = set(
    df.loc[X_val.index, "URL"]
)

test_urls = set(
    df.loc[X_test.index, "URL"]
)

train_val_overlap = train_urls.intersection(
    val_urls
)

train_test_overlap = train_urls.intersection(
    test_urls
)

val_test_overlap = val_urls.intersection(
    test_urls
)

print(
    "Train-Val overlap:",
    len(train_val_overlap)
)

print(
    "Train-Test overlap:",
    len(train_test_overlap)
)

print(
    "Val-Test overlap:",
    len(val_test_overlap)
)

# Memastikan tidak ada url yang tumpang tindih
assert train_urls.isdisjoint(val_urls)
assert train_urls.isdisjoint(test_urls)
assert val_urls.isdisjoint(test_urls)

print("\nTidak terdapat overlap URL antarsubset.")

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

Tidak terdapat overlap URL antarsubset.


In [ ]:
# ngecek apakah terdapat baris dengan nilai fitur yang sama, meskipun url nya berbeda

print(
    "Jumlah duplikat berdasarkan seluruh fitur:",
    X.duplicated().sum()
)

print(
    "Jumlah duplikat fitur dan label:",
    pd.concat([X, y], axis=1).duplicated().sum()
)

Jumlah duplikat berdasarkan seluruh fitur: 0
Jumlah duplikat fitur dan label: 0


In [ ]:
# gecek apakah kombinasi fitur yang sama

train_feature_rows = set(
    map(tuple, X_train.to_numpy())
)

val_feature_rows = set(
    map(tuple, X_val.to_numpy())
)

test_feature_rows = set(
    map(tuple, X_test.to_numpy())
)

train_val_feature_overlap = (
    train_feature_rows.intersection(val_feature_rows)
)

train_test_feature_overlap = (
    train_feature_rows.intersection(test_feature_rows)
)

val_test_feature_overlap = (
    val_feature_rows.intersection(test_feature_rows)
)

print(
    "Overlap fitur Train-Val:",
    len(train_val_feature_overlap)
)

print(
    "Overlap fitur Train-Test:",
    len(train_test_feature_overlap)
)

print(
    "Overlap fitur Val-Test:",
    len(val_test_feature_overlap)
)

Overlap fitur Train-Val: 0
Overlap fitur Train-Test: 0
Overlap fitur Val-Test: 0


In [ ]:
# memastikan semua data masuk tepat ke salah satu subset
total_split = (
    len(X_train) +
    len(X_val) +
    len(X_test)
)

print("Jumlah data asli :", len(X))
print("Jumlah hasil split:", total_split)

assert total_split == len(X)

all_indices = (
    set(X_train.index) |
    set(X_val.index) |
    set(X_test.index)
)

assert len(all_indices) == len(X)

print("Seluruh data telah terbagi tanpa hilang atau terduplikasi.")

Jumlah data asli : 234562
Jumlah hasil split: 234562
Seluruh data telah terbagi tanpa hilang atau terduplikasi.
